In [7]:
from sgp4.api import Satrec
from skyfield.api import EarthSatellite, load
import datetime
import pandas as pd
import numpy as np

In [2]:
# TO TRZEBA PODMIENIĆ ŻEBY ZAMIAST DNI BYLY GODZINY, CO 24H żeby zrobić 10dni, pierwszy pomiar ma być najlbliższe x godzin do godziny 12.00 żeby odnosić się do ground-truth z sp3, a kolejne co 24h od pierwszego

import pandas as pd
import datetime
from skyfield.api import EarthSatellite, load
from skyfield.framelib import itrs

INPUT_CSV = 'tle_C38_2023-25.csv'
OUTPUT_CSV = 'ECEF_C38_2023-25_long_format-10perEpoch.csv'

SAT_ID = 'PC38' 

ts = load.timescale()
df = pd.read_csv(INPUT_CSV) 

w = []
day_offset = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

for index, row in df.iterrows():
    l1 = row['TLE_LINE1'] 
    l2 = row['TLE_LINE2'] 
    tle_epoch_str = row['EPOCH']
    
    try:
        sat = EarthSatellite(l1, l2, SAT_ID, ts)
        dt_epoka = sat.epoch.utc_datetime()
        bstar = sat.model.bstar
        
        for offset in day_offset:
            # 1. Wyznaczenie daty docelowej (12:00 + offset)
            dt_target = (dt_epoka + datetime.timedelta(days=offset)).replace(hour=12, minute=0, second=0, microsecond=0)
            
            # 2. Korekta -18 sekund pod pliki SP3 (czas GPS)
            dt_target_utc = dt_target - datetime.timedelta(seconds=18)
            t_cel = ts.from_datetime(dt_target_utc)
            
            # 3. Kinematyka w układzie ECEF
            geocentric = sat.at(t_cel)
            pos_, vel_ = geocentric.frame_xyz_and_velocity(itrs)
            pos_ecef = pos_.km
            vel_ecef = vel_.km_per_s
            
            # Tworzymy osobny wiersz dla każdego przesunięcia
            record = {
                'tle_epoch': tle_epoch_str,
                'target_epoch': dt_target.strftime('%Y-%m-%d %H:%M:%S'), # Klucz do złączenia z SP3
                'sat_id': SAT_ID,
                'days_offset': offset,
                'bstar': bstar,
                'x': round(pos_ecef[0], 6),
                'y': round(pos_ecef[1], 6),
                'z': round(pos_ecef[2], 6),
                'vx': round(vel_ecef[0], 6),
                'vy': round(vel_ecef[1], 6),
                'vz': round(vel_ecef[2], 6)
            }
            
            w.append(record)
            
    except Exception as ex:
        print(f"Błąd wiersza {index}: {ex}")

df_wyniki = pd.DataFrame(w)
df_wyniki.to_csv(OUTPUT_CSV, index=False)

print(f"Zapisano wyniki w osobrnych wierszach: {OUTPUT_CSV}")

Zapisano wyniki w osobrnych wierszach: ECEF_C38_2023-25_long_format-10perEpoch.csv


In [3]:
df_sgp4 = pd.read_csv("ECEF_C38_2023-25_long_format-10perEpoch.csv")
df_sp3 = pd.read_csv("sp3_C38_2023-25.csv")

df_final = pd.merge(df_sgp4, df_sp3, left_on=["target_epoch", "sat_id"], right_on=["epoch", "sat_id"], how="inner")

df_final.to_csv("final.csv", index=False, encoding='utf-8')

In [4]:
df = pd.read_csv("final.csv")

df = df.rename(columns={
    'x_x': 'x', 'y_x': 'y', 'z_x': 'z',
    'x_y': 'sp3_x', 'y_y': 'sp3_y', 'z_y': 'sp3_z'
})
df.to_csv("final.csv", index=False, encoding='utf-8')

In [5]:
df = pd.read_csv("final.csv")
df['x_error'] = df['sp3_x'] - df['x']
df['y_error'] = df['sp3_y'] - df['y']
df['z_error'] = df['sp3_z'] - df['z']

columns_to_keep = [
    'tle_epoch', 'target_epoch', 'days_offset', 'bstar', 'x', 'y', 'z', 'vx', 'vy', 'vz', 'x_error', 'y_error', 'z_error'
    ]

df_final = df[columns_to_keep]

df_final.to_csv("C38_ready-forSure_v2.csv", index=False, encoding='utf-8')

In [12]:
ds1 = pd.read_csv("tle_C38_2023_onePerDay.csv")
ds2 = pd.read_csv("tle_C38_2024-25.csv")

merged = pd.concat([ds1, ds2], ignore_index=True)
merged.to_csv("tle_C38_2023-25.csv", index=False, encoding='utf-8')

In [13]:
df = pd.read_csv("tle_C38_2023-25.csv")

col_date = "EPOCH"

df = (
    df
    .drop_duplicates()
    .sort_values(by=col_date, key=pd.to_datetime)
    .reset_index(drop=True)
)

df.to_csv("tle_C38_2023-25.csv", index=False, encoding="utf-8")

In [8]:
#Clearing outliers
err3d = np.linalg.norm(df[['x_error', 'y_error', 'z_error']], axis=1)
clean = df[err3d <= 50.0].reset_index(drop=True)

clean.to_csv("cleaned_final.csv", index=False, encoding="utf-8")